# XGBoost Training — ATUS Television Dataset

Trains an XGBoost binary classifier for **TV-watching prediction** on ATUS per-respondent hourly data.

Mirrors the Plegma/REFIT structure as closely as possible, with two differences:
- **Features:** weather excluded (ATUS has no real weather); `is_evening` added instead of `month`
- **Groups:** respondent ID (`tucaseid`) used instead of house ID for GroupKFold CV

> `tv_minutes` and `month` are excluded to prevent data leakage (both are derived from/correlated with the target).

## Configuration

In [1]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score,
                             roc_auc_score, classification_report)

warnings.filterwarnings("ignore")

# ── 1. Config ─────────────────────────────────────────────────────────────────
INPUT_PATH  = r"C:\Users\moham\Documents\490 project new\atus_tv_hourly.csv"
MODELS_DIR  = r"C:\Users\moham\Documents\490 project new\models_atus"
RANDOM_SEED = 42

TARGET = 'elec_television_on'

# Month excluded — always January due to synthetic timestamp
# tv_minutes excluded — derived from target, would cause leakage
FEATURE_COLS = [
    'hour',
    'day_of_week',
    'is_weekend',
    'is_evening',
]

PARAM_GRID = {
    "n_estimators":  [100, 300],
    "max_depth":     [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.3],
}

## Load & Inspect Data

In [2]:
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Loading {INPUT_PATH} ...")
df = pd.read_csv(INPUT_PATH, dtype={'tucaseid': str})

# Drop columns that are useless or would cause leakage
drop_cols = ['tv_minutes', 'month', 'timestamp']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

print(f"  Rows:        {len(df):,}")
print(f"  Respondents: {df['tucaseid'].nunique():,}")
print(f"  Features:    {FEATURE_COLS}")
print(f"  Target ON rate: {df[TARGET].mean():.2%}\n")

Loading C:\Users\moham\Documents\490 project new\atus_tv_hourly.csv ...
  Rows:        3,727,872
  Respondents: 155,328
  Features:    ['hour', 'day_of_week', 'is_weekend', 'is_evening']
  Target ON rate: 24.27%



## Time-Based Train/Test Split

In [3]:
# Sort by respondent then hour so the split is consistent
df = df.sort_values(['tucaseid', 'hour']).reset_index(drop=True)

split      = int(len(df) * 0.8)
train_mask = df.index < split
test_mask  = df.index >= split

X_train = df.loc[train_mask, FEATURE_COLS].reset_index(drop=True)
X_test  = df.loc[test_mask,  FEATURE_COLS].reset_index(drop=True)
y_train = df.loc[train_mask, TARGET].astype(int).reset_index(drop=True)
y_test  = df.loc[test_mask,  TARGET].astype(int).reset_index(drop=True)
groups  = df.loc[train_mask, 'tucaseid'].reset_index(drop=True)

on_train = y_train.sum()
on_test  = y_test.sum()
print(f"  Train rows: {len(X_train):,} | ON: {on_train:,}")
print(f"  Test  rows: {len(X_test):,}  | ON: {on_test:,}\n")

  Train rows: 2,982,297 | ON: 717,493
  Test  rows: 745,575  | ON: 187,296



## Model Training with GroupKFold Grid Search

In [4]:
print(f"{'='*55}")
print(f"APPLIANCE: {TARGET}")
print(f"{'='*55}")

neg = (y_train == 0).sum()
pos = on_train
scale_pos_weight = min(neg / pos, 5.0) if pos > 0 else 1.0
print(f"  scale_pos_weight: {scale_pos_weight:.2f}")

base_model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=RANDOM_SEED,
    device="cuda",
    n_jobs=-1,
)

n_splits    = min(5, df.loc[train_mask, 'tucaseid'].nunique())
group_kfold = GroupKFold(n_splits=n_splits)
cv_splits   = group_kfold.split(X_train, y_train, groups=groups)

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=PARAM_GRID,
    cv=cv_splits,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=0,
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_


APPLIANCE: elec_television_on
  scale_pos_weight: 3.16


## Evaluation & Model Saving

In [5]:
y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, zero_division=0)
try:
    auc = roc_auc_score(y_test, y_proba)
except ValueError:
    auc = float("nan")

print(f"\n  Best params: {grid_search.best_params_}")
print(f"  CV F1:       {grid_search.best_score_:.4f}")
print(f"  Test Acc:    {acc:.4f} | Test F1: {f1:.4f} | AUC: {auc:.4f}")
print(classification_report(y_test, y_pred, zero_division=0))

# ── Save ───────────────────────────────────────────────────────────────────
model_path = os.path.join(MODELS_DIR, f"{TARGET}.pkl")
with open(model_path, "wb") as f:
    pickle.dump(best_model, f)
print(f"  Saved → {model_path}")


  Best params: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300}
  CV F1:       0.6492
  Test Acc:    0.6934 | Test F1: 0.5410 | AUC: 0.7785
              precision    recall  f1-score   support

           0       0.88      0.68      0.77    558279
           1       0.43      0.72      0.54    187296

    accuracy                           0.69    745575
   macro avg       0.66      0.70      0.66    745575
weighted avg       0.77      0.69      0.71    745575

  Saved → C:\Users\moham\Documents\490 project new\models_atus\elec_television_on.pkl
